In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# Load the saved feature dataset
df = pd.read_csv(r"C:\Users\moham\OneDrive\Desktop\KFUPM\Courses\242\AML\Project\ImageDataset\image_features.csv")

# Preview
print(df.head())

             ImageName        Label  mobilenetv2_feature_1  \
0    Anserverbot_1.png  Anserverbot                    0.0   
1   Anserverbot_10.png  Anserverbot                    0.0   
2  Anserverbot_100.png  Anserverbot                    0.0   
3  Anserverbot_101.png  Anserverbot                    0.0   
4  Anserverbot_102.png  Anserverbot                    0.0   

   mobilenetv2_feature_2  mobilenetv2_feature_3  mobilenetv2_feature_4  \
0                    0.0                    0.0                    0.0   
1                    0.0                    0.0                    0.0   
2                    0.0                    0.0                    0.0   
3                    0.0                    0.0                    0.0   
4                    0.0                    0.0                    0.0   

   mobilenetv2_feature_5  mobilenetv2_feature_6  mobilenetv2_feature_7  \
0                    0.0                    0.0                    0.0   
1                    0.0          

In [5]:
# Define 5 popular classifiers
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
}

In [6]:
from collections import defaultdict

# Store results
results = defaultdict(dict)

# Get unique CNN prefixes from column names
cnn_prefixes = set(col.split('_feature_')[0] for col in df.columns if '_feature_' in col)

# Encode class labels as integers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["LabelEncoded"] = le.fit_transform(df["Label"])

# Loop over each CNN feature set
for cnn in sorted(cnn_prefixes):
    print(f"\n=== Using features from {cnn} ===")
    
    # Extract features for current CNN
    feature_cols = [col for col in df.columns if col.startswith(cnn)]
    X = df[feature_cols].values
    y = df["LabelEncoded"].values

    # Normalize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    for clf_name, clf in classifiers.items():
        print(f"Training {clf_name}...")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # Save results
        results[cnn][clf_name] = acc
        print(f"{clf_name} Accuracy on {cnn}: {acc:.4f}")



=== Using features from densenet121 ===
Training LogisticRegression...


c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Accuracy on densenet121: 0.8796
Training SVM...
SVM Accuracy on densenet121: 0.8827
Training RandomForest...
RandomForest Accuracy on densenet121: 0.9105
Training KNN...
KNN Accuracy on densenet121: 0.8241

=== Using features from efficientnet_b0 ===
Training LogisticRegression...


c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Accuracy on efficientnet_b0: 0.8981
Training SVM...
SVM Accuracy on efficientnet_b0: 0.8704
Training RandomForest...
RandomForest Accuracy on efficientnet_b0: 0.9074
Training KNN...
KNN Accuracy on efficientnet_b0: 0.8241

=== Using features from mobilenetv2 ===
Training LogisticRegression...


c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Accuracy on mobilenetv2: 0.9043
Training SVM...
SVM Accuracy on mobilenetv2: 0.8611
Training RandomForest...
RandomForest Accuracy on mobilenetv2: 0.8796
Training KNN...
KNN Accuracy on mobilenetv2: 0.7963

=== Using features from resnet18 ===
Training LogisticRegression...
LogisticRegression Accuracy on resnet18: 0.8704
Training SVM...
SVM Accuracy on resnet18: 0.8704
Training RandomForest...
RandomForest Accuracy on resnet18: 0.9043
Training KNN...
KNN Accuracy on resnet18: 0.8395

=== Using features from vgg16 ===
Training LogisticRegression...
LogisticRegression Accuracy on vgg16: 0.8827
Training SVM...
SVM Accuracy on vgg16: 0.8642
Training RandomForest...
RandomForest Accuracy on vgg16: 0.8858
Training KNN...
KNN Accuracy on vgg16: 0.7994


In [7]:
# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results).T  # Transpose so CNNs are rows, classifiers are columns
print("\n=== Summary of All Classifier Accuracies ===")
print(results_df)

# Optionally save
results_df.to_csv("ml_classifier_results_per_cnn.csv")



=== Summary of All Classifier Accuracies ===
                 LogisticRegression       SVM  RandomForest       KNN
densenet121                0.879630  0.882716      0.910494  0.824074
efficientnet_b0            0.898148  0.870370      0.907407  0.824074
mobilenetv2                0.904321  0.861111      0.879630  0.796296
resnet18                   0.870370  0.870370      0.904321  0.839506
vgg16                      0.882716  0.864198      0.885802  0.799383


In [8]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For consistent layout
import math

In [10]:
from sklearn.metrics import confusion_matrix

# Reset results storage
confusion_matrices = defaultdict(dict)

# Loop over each CNN feature set again
for cnn in sorted(cnn_prefixes):
    print(f"\n\n===== CNN Features: {cnn} =====")
    
    # Extract features
    feature_cols = [col for col in df.columns if col.startswith(cnn)]
    X = df[feature_cols].values
    y = df["LabelEncoded"].values

    # Normalize
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    for clf_name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        cm = confusion_matrix(y_test, y_pred)
        confusion_matrices[cnn][clf_name] = cm

        print(f"\n--- {clf_name} ---")
        print(cm)



===== CNN Features: densenet121 =====


c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- LogisticRegression ---
[[41  0  1  0  0  0  0  0  0  1  1  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 64  1  0  3  0  0  0  2  0  1  0]
 [ 2  0  6 39  0  2  0  0  0  0  0  0  0]
 [ 0  0  1  0 17  0  0  0  0  0  2  0  0]
 [ 0  0  4  1  0 32  0  0  0  2  0  0  0]
 [ 0  0  0  0  0  0 15  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  5  0  0  0  0]
 [ 0  0  2  0  0  3  0  1  0  3  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 19  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0 13]]

--- SVM ---
[[41  0  2  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 67  2  1  1  0  0  0  0  0  0  0]
 [ 2  0  4 42  0  1  0  0  0  0  0  0  0]
 [ 2  0  0  2 15  1  0  0  0  0  0  0  0]
 [ 0  0  6  1  0 32  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0 14  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  4  0  0  0  0]
 [ 0  0  4  0  0  0  0  0  0  5  0

c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- LogisticRegression ---
[[41  0  1  0  0  2  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 64  2  2  3  0  0  0  1  0  0  0]
 [ 0  0  5 40  0  2  0  0  1  0  0  0  1]
 [ 0  0  0  0 20  0  0  0  0  0  0  0  0]
 [ 0  0  3  1  0 35  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 15  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  5  0  0  0  0]
 [ 0  0  3  0  0  2  0  0  0  3  0  0  1]
 [ 0  0  0  0  0  0  0  0  0  0 19  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0]
 [ 0  0  0  0  1  1  0  0  0  0  0  0 12]]

--- SVM ---
[[42  0  1  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 65  4  0  2  0  0  0  0  0  0  0]
 [ 2  0  3 43  0  1  0  0  0  0  0  0  0]
 [ 3  0  1  1 15  0  0  0  0  0  0  0  0]
 [ 0  0  3  3  0 33  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0 14  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  1  0  0  0  0  4  0  0  0  0]
 [ 0  0  4  1  0  1  0  0  0  3  0

KeyboardInterrupt: 